# Notebook to work on the API to get data from EIA 

In [3]:
#this uses a key so need to load said key to use
import os
from dotenv import load_dotenv

load_dotenv() #load the .env file
API_KEY = os.getenv("EIA_API_KEY") #get the key from the .env file

#make sure key is there - don't want to print key to console so just check if it's there
print(API_KEY is not None) #True if it has something, False if it is None. Quick check that it loads something

True


So far looking good, now can use it pull data from the api

In [8]:
import requests


resp = requests.get("https://api.eia.gov/v2/electricity/rto/region-data/",
                    params={"api_key": API_KEY}, timeout=30)
resp.json()

{'response': {'id': 'region-data',
  'name': 'Hourly Demand, Demand Forecast, Generation, and Interchange',
  'description': 'Hourly demand, day-ahead demand forecast, net generation, and interchange by balancing authority.  \n    Source: Form EIA-930\n    Product: Hourly Electric Grid Monitor',
  'frequency': [{'id': 'hourly',
    'alias': 'hourly (UTC)',
    'description': 'One data point for each hour in UTC time.',
    'query': 'H',
    'format': 'YYYY-MM-DD"T"HH24'},
   {'id': 'local-hourly',
    'alias': 'hourly (Local Time Zone)',
    'description': 'One data point for each hour in local time.',
    'query': 'LH',
    'format': 'YYYY-MM-DD"T"HH24TZH'}],
  'facets': [{'id': 'respondent',
    'description': 'Balancing Authority / Region'},
   {'id': 'type', 'description': 'Metric'}],
  'data': {'value': {'aggregation-method': 'SUM',
    'alias': 'Demand, Demand Forecast, Net Generation, or Interchange',
    'units': 'megawatthours'}},
  'startPeriod': '2019-01-01T00',
  'endPeriod

In [9]:
#shorten and make easier to read the endpoint for the data itself
BASE_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data" #extra data endpoint for the data itself

params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "BPAT", #coming from api at epi, can change to other regions if desired
    "facets[type][]":["D","DF"],
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000,
}

resp = requests.get(BASE_URL, params=params, timeout=30)
resp.status_code #check if the request was successful, 200 means success

200

In [16]:
resp.json()

{'warnings': [{'warning': 'incomplete return',
   'description': 'The API can only return 5000 rows in JSON format.  Please consider constraining your request with facet, start, or end, or using offset to paginate results.'}],
 'response': {'total': '135800',
  'dateFormat': 'YYYY-MM-DD"T"HH24',
  'frequency': 'hourly',
  'data': [{'period': '2026-10-02T07',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'DF',
    'type-name': 'Day-ahead demand forecast',
    'value': '6099',
    'value-units': 'megawatthours'},
   {'period': '2026-10-02T06',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'DF',
    'type-name': 'Day-ahead demand forecast',
    'value': '6425',
    'value-units': 'megawatthours'},
   {'period': '2026-10-02T05',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'DF',
    'type-name': 'Day-ahead demand forecast',
    'value': '676

Warnings here are helpful, I didn't limit by star or end and just used the 5000 from the website, which is what it uses to limit calls. So control time frame with start and end or use the offset to move times as it states. 

In [ ]:
#check the data
body = resp.json()["response"]
body.keys() #check the keys in the response body

dict_keys(['total', 'dateFormat', 'frequency', 'data', 'description'])

In [18]:
{k: v for k, v in body.items() if k != "data"}

{'total': '135800',
 'dateFormat': 'YYYY-MM-DD"T"HH24',
 'frequency': 'hourly',
 'description': 'Hourly demand, day-ahead demand forecast, net generation, and interchange by balancing authority.  \n    Source: Form EIA-930\n    Product: Hourly Electric Grid Monitor'}

In [13]:
print(body['total'])
print(body['data'][:5]) #print the first 5 rows of data to see what it looks like

135800
[{'period': '2026-10-02T07', 'respondent': 'BPAT', 'respondent-name': 'Bonneville Power Administration', 'type': 'DF', 'type-name': 'Day-ahead demand forecast', 'value': '6099', 'value-units': 'megawatthours'}, {'period': '2026-10-02T06', 'respondent': 'BPAT', 'respondent-name': 'Bonneville Power Administration', 'type': 'DF', 'type-name': 'Day-ahead demand forecast', 'value': '6425', 'value-units': 'megawatthours'}, {'period': '2026-10-02T05', 'respondent': 'BPAT', 'respondent-name': 'Bonneville Power Administration', 'type': 'DF', 'type-name': 'Day-ahead demand forecast', 'value': '6762', 'value-units': 'megawatthours'}, {'period': '2026-10-02T04', 'respondent': 'BPAT', 'respondent-name': 'Bonneville Power Administration', 'type': 'DF', 'type-name': 'Day-ahead demand forecast', 'value': '7060', 'value-units': 'megawatthours'}, {'period': '2026-10-02T03', 'respondent': 'BPAT', 'respondent-name': 'Bonneville Power Administration', 'type': 'DF', 'type-name': 'Day-ahead demand for

things live in data, and all things appear to be strings so will need to convert if needing something else, for example value should be an integer

In [14]:
import pandas as pd #now load dataframe

df = pd.DataFrame(body["data"])

#convert values as mentioned, 
df["period"] = pd.to_datetime(df["period"], format="%Y-%m-%dT%H", utc=True)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

In [ ]:
len(df) #check the length of the dataframe, should be 5000 as requested

5000

In [19]:
df['period'].min(), df['period'].max() #check the min and max dates in the dataframe

(Timestamp('2026-06-19 00:00:00+0000', tz='UTC'),
 Timestamp('2026-10-02 07:00:00+0000', tz='UTC'))

In [21]:
#going to give start and end dates to get a specific range of data, 


params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "BPAT", #coming from api at epi, can change to other regions if desired
    "facets[type][]":["D","DF"],
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000,
    "start": "2026-01-12T00",
    "end": "2026-01-14T23",
}

resp = requests.get(BASE_URL, params=params, timeout=30)
resp.status_code #check if the request was successful, 200 means success

200

In [22]:
resp.json() #check the response, should be a dictionary with keys like 'response', 'status', etc.

{'response': {'total': '144',
  'dateFormat': 'YYYY-MM-DD"T"HH24',
  'frequency': 'hourly',
  'data': [{'period': '2026-01-14T23',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'D',
    'type-name': 'Demand',
    'value': '7612',
    'value-units': 'megawatthours'},
   {'period': '2026-01-14T23',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'DF',
    'type-name': 'Day-ahead demand forecast',
    'value': '7458',
    'value-units': 'megawatthours'},
   {'period': '2026-01-14T22',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'D',
    'type-name': 'Demand',
    'value': '7710',
    'value-units': 'megawatthours'},
   {'period': '2026-01-14T22',
    'respondent': 'BPAT',
    'respondent-name': 'Bonneville Power Administration',
    'type': 'DF',
    'type-name': 'Day-ahead demand forecast',
    'value': '7496',
    'value-units': 'megawatth

No warnings this time and total is less than before

In [23]:
body = resp.json()["response"]
body.keys() #check the keys in the response body

dict_keys(['total', 'dateFormat', 'frequency', 'data', 'description'])

In [29]:
df = pd.DataFrame(body["data"])
df.head()


,period,respondent,respondent-name,type,type-name,value,value-units
0,2026-01-14T23,BPAT,Bonneville Power Administration,D,Demand,7612,megawatthours
1,2026-01-14T23,BPAT,Bonneville Power Administration,DF,Day-ahead demand forecast,7458,megawatthours
2,2026-01-14T22,BPAT,Bonneville Power Administration,D,Demand,7710,megawatthours
3,2026-01-14T22,BPAT,Bonneville Power Administration,DF,Day-ahead demand forecast,7496,megawatthours
4,2026-01-14T21,BPAT,Bonneville Power Administration,D,Demand,7855,megawatthours


In [30]:
#convert values as mentioned, 
df["period"] = pd.to_datetime(df["period"], format="%Y-%m-%dT%H", utc=True)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

In [ ]:
len(df) #check the length of the dataframe, should be 144 since that is total found in the response body

144

In [ ]:
df.columns

Index(['period', 'respondent', 'respondent-name', 'type', 'type-name', 'value',
       'value-units'],
      dtype='str')

In [ ]:
df.pivot(index="period", columns="type", values="value").head() #make pivot table to see the data in a more readable format, 
                                                                #with period as index and type as columns

type,D,DF
period,,
2026-01-12 00:00:00+00:00,7806,7836
2026-01-12 01:00:00+00:00,8112,8093
2026-01-12 02:00:00+00:00,8407,8341
2026-01-12 03:00:00+00:00,8368,8269
2026-01-12 04:00:00+00:00,8243,8170


Shows D and DF side by side and what they saw for a given period. 

To handle all data would need to loop through with offset, this would be easiest and keep adding to make final dataframe

In [32]:
#test the loop through and add
import time #to sleep so we don't spam requests to the server


params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "BPAT", #coming from api at epi, can change to other regions if desired
    "facets[type][]":["D","DF"],
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000,
    "start": "2025-01-12T00", #this should give us a lot more data, but we will loop through to get it all
    "end": "2026-01-14T23",
}

#pagination loop: keep asking for the next 5000 rows until we have them all
all_rows = []
offset = 0 #start, update as we go through the loop

while True:
    params["offset"] = offset
    resp = requests.get(BASE_URL, params=params, timeout=30)
    body = resp.json()["response"]

    page = body["data"]
    all_rows.extend(page) #add on
    offset += len(page) #go to end of the page we just got, so next time we get the next page

    #check progress, print out how many rows we have and how many total there are
    total = int(body["total"]) #EIA sends total as a string
    print(f"  {offset:,} / {total:,} rows")

    #stop when we have everything (or a page comes back empty)
    if len(page) == 0 or offset >= total:
        break

    time.sleep(1) #be polite to the server

  5,000 / 17,664 rows
  10,000 / 17,664 rows
  15,000 / 17,664 rows
  17,664 / 17,664 rows


In [33]:
df = pd.DataFrame(all_rows)

expected = len(pd.date_range("2025-01-12T00", "2026-01-14T23", freq="h")) * 2
print(len(df), expected)
print(df.duplicated(subset=["period", "type"]).sum())   #should be 0

17664 17664
0


Cool so have things working, now build function 

In [34]:
BASE_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

def fetch_demand(start: str, end: str, 
                 respondent: str = "BPAT", types: list = None):

    #check if given or set to default
    if types is None:
        types = ["D", "DF"]

    #build params for the request
    params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": respondent,
    "facets[type][]":types,
    "sort[0][column]": "period",
    "sort[0][direction]": "asc", #ascending order 
    "offset": 0,
    "length": 5000,
    "start": start, 
    "end": end,
}

    #page through the results 5000 rows at a time
    all_rows = []
    offset = 0
    while True:
        params["offset"] = offset
        resp = requests.get(BASE_URL, params=params, timeout=60)

        #not using raise_for_status(), its error message prints the url which has the api key
        if resp.status_code != 200:
            raise Exception(f"Request failed with status code {resp.status_code}: {resp.text[:200]}")

        body = resp.json()["response"]
        page = body["data"]
        all_rows.extend(page)
        offset += len(page)
        print(f"{offset} / {body['total']}")

        if len(page) == 0 or offset >= int(body["total"]):
            break

    #make dataframe and fix types
    df = pd.DataFrame(all_rows)
    df["period"] = pd.to_datetime(df["period"], format="%Y-%m-%dT%H", utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

In [35]:
#now test the function to make sure it works
df = fetch_demand("2025-10-12T00", "2026-01-14T23")

4560 / 4560
